# Fine-tune XTTS v2 on `audio-kri-russian` (Kaggle T4)

Fine-tunes **Coqui XTTS v2** on the Russian lecturer dataset and pushes the
trained model back to the HF Hub for CPU inference.

**Before running:**
1. Notebook settings → Accelerator → **GPU T4 x1**, and **Internet: ON**.
2. Add your HF token as a Kaggle Secret named **`HF_TOKEN`**.
3. Make sure the dataset `audio-kri-russian` exists on your HF account
   (produced by the transcription notebook).

**License note:** XTTS v2 is **non-commercial (CPML)** — fine for R&D, not for a product.


In [ ]:
# 1. Install Coqui TTS (maintained fork) + deps
!pip install -q coqui-tts "datasets>=2.18" soundfile huggingface_hub


In [ ]:
# 2. HF login (Kaggle Secrets)
import os
from huggingface_hub import login, whoami

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN not found. Add it under Add-ons -> Secrets."
login(token=HF_TOKEN)

USERNAME       = whoami()["name"]
DATASET_REPO   = f"{USERNAME}/audio-kri-russian"     # source dataset
MODEL_REPO     = f"{USERNAME}/xtts-v2-kri-russian"   # output model
LANGUAGE       = "ru"
print("Dataset:", DATASET_REPO, "| Model out:", MODEL_REPO)


In [ ]:
# 3. Pull dataset, write wavs (resampled to 22.05k) + LJSpeech metadata
import os, soundfile as sf, numpy as np
from datasets import load_dataset, Audio

DATA_DIR = "/kaggle/working/dataset"
WAV_DIR  = os.path.join(DATA_DIR, "wavs")
os.makedirs(WAV_DIR, exist_ok=True)

# XTTS expects 22050 Hz training audio. Source is 16k mono -> upsample for compat.
ds = load_dataset(DATASET_REPO, split="train")
ds = ds.cast_column("audio", Audio(sampling_rate=22050))

rows = []
for i, ex in enumerate(ds):
    clip_id = f"clip_{i+1:04d}"
    wav = np.asarray(ex["audio"]["array"], dtype="float32")
    sf.write(os.path.join(WAV_DIR, clip_id + ".wav"), wav, 22050)
    text = ex["text"].replace("|", " ").replace("\n", " ").strip()
    rows.append((clip_id, text))

print(f"Wrote {len(rows)} wavs to {WAV_DIR}")

# 99/1 train/eval split; LJSpeech format: id|text|text
split = max(1, int(len(rows) * 0.01))
eval_rows, train_rows = rows[:split], rows[split:]

def write_meta(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for cid, txt in items:
            f.write(f"{cid}|{txt}|{txt}\n")

write_meta(os.path.join(DATA_DIR, "metadata_train.csv"), train_rows)
write_meta(os.path.join(DATA_DIR, "metadata_eval.csv"), eval_rows)
print(f"train={len(train_rows)}  eval={len(eval_rows)}")

# Keep one wav as the speaker reference for eval/inference
SPEAKER_REFERENCE = [os.path.join(WAV_DIR, train_rows[0][0] + ".wav")]
print("Speaker reference:", SPEAKER_REFERENCE)


In [ ]:
# 4. Download XTTS v2 base checkpoints
import os
from TTS.utils.manage import ModelManager

OUT_PATH = "/kaggle/working/xtts_finetune"
CKPT_DIR = os.path.join(OUT_PATH, "XTTS_v2.0_original_model_files")
os.makedirs(CKPT_DIR, exist_ok=True)

FILES = {
    "dvae.pth":      "https://huggingface.co/coqui/XTTS-v2/resolve/main/dvae.pth",
    "mel_stats.pth": "https://huggingface.co/coqui/XTTS-v2/resolve/main/mel_stats.pth",
    "vocab.json":    "https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json",
    "model.pth":     "https://huggingface.co/coqui/XTTS-v2/resolve/main/model.pth",
    "config.json":   "https://huggingface.co/coqui/XTTS-v2/resolve/main/config.json",
}
to_get = [u for n, u in FILES.items() if not os.path.isfile(os.path.join(CKPT_DIR, n))]
if to_get:
    ModelManager._download_model_files(to_get, CKPT_DIR, progress_bar=True)

DVAE_CHECKPOINT = os.path.join(CKPT_DIR, "dvae.pth")
MEL_NORM_FILE   = os.path.join(CKPT_DIR, "mel_stats.pth")
TOKENIZER_FILE  = os.path.join(CKPT_DIR, "vocab.json")
XTTS_CHECKPOINT = os.path.join(CKPT_DIR, "model.pth")
print("Base checkpoints ready.")


In [ ]:
# 5. Configure the GPT XTTS trainer
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.layers.xtts.trainer.gpt_trainer import (
    GPTArgs, GPTTrainer, GPTTrainerConfig, XttsAudioConfig,
)

RUN_NAME, PROJECT_NAME = "GPT_XTTS_kri", "XTTS_kri"
EPOCHS          = 12      # watch eval loss; bump if still improving
BATCH_SIZE      = 3       # fits T4 16GB
GRAD_ACUMM_STEPS= 84      # effective batch ~252

config_dataset = BaseDatasetConfig(
    formatter="ljspeech",
    dataset_name="kri",
    path=DATA_DIR,
    meta_file_train="metadata_train.csv",
    meta_file_val="metadata_eval.csv",
    language=LANGUAGE,
)

model_args = GPTArgs(
    max_conditioning_length=132300,
    min_conditioning_length=66150,
    max_wav_length=255995,         # ~11.6s at 22050
    max_text_length=200,
    mel_norm_file=MEL_NORM_FILE,
    dvae_checkpoint=DVAE_CHECKPOINT,
    xtts_checkpoint=XTTS_CHECKPOINT,
    tokenizer_file=TOKENIZER_FILE,
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)
audio_config = XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000)

config = GPTTrainerConfig(
    output_path=OUT_PATH,
    model_args=model_args,
    run_name=RUN_NAME,
    project_name=PROJECT_NAME,
    run_description="XTTS v2 fine-tune on Russian lecturer",
    dashboard_logger="tensorboard",
    audio=audio_config,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    batch_group_size=48,
    eval_batch_size=BATCH_SIZE,
    num_loader_workers=4,
    print_step=50, plot_step=100, log_model_step=1000,
    save_step=1000, save_n_checkpoints=1, save_checkpoints=True,
    print_eval=False,
    optimizer="AdamW",
    optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=5e-06,
    lr_scheduler="MultiStepLR",
    lr_scheduler_params={"milestones": [900000, 2700000, 5400000], "gamma": 0.5, "last_epoch": -1},
    test_sentences=[
        {"text": "Здравствуйте, сегодня мы поговорим о криминалистике.",
         "speaker_wav": SPEAKER_REFERENCE, "language": LANGUAGE},
    ],
)
print("Config ready.")


In [ ]:
# 6. Train
from trainer import Trainer, TrainerArgs
from TTS.tts.datasets import load_tts_samples

model = GPTTrainer.init_from_config(config)
train_samples, eval_samples = load_tts_samples(
    [config_dataset], eval_split=True, eval_split_max_size=256, eval_split_size=0.01,
)
print(f"train={len(train_samples)} eval={len(eval_samples)}")

trainer = Trainer(
    TrainerArgs(restore_path=None, skip_train_epoch=False,
                start_with_eval=True, grad_accum_steps=GRAD_ACUMM_STEPS),
    config, output_path=OUT_PATH, model=model,
    train_samples=train_samples, eval_samples=eval_samples,
)
trainer.fit()
print("Training done. Output run dir:", trainer.output_path)


In [ ]:
# 7. Collect the fine-tuned model files and push to HF
import os, glob, shutil
from huggingface_hub import HfApi

run_dir = trainer.output_path  # e.g. /kaggle/working/xtts_finetune/GPT_XTTS_kri-<date>
EXPORT = "/kaggle/working/xtts_kri_export"
os.makedirs(EXPORT, exist_ok=True)

# best model checkpoint
best = sorted(glob.glob(os.path.join(run_dir, "best_model*.pth")))
ckpt = best[-1] if best else os.path.join(run_dir, "best_model.pth")
shutil.copy(ckpt, os.path.join(EXPORT, "model.pth"))
shutil.copy(os.path.join(run_dir, "config.json"), os.path.join(EXPORT, "config.json"))
shutil.copy(TOKENIZER_FILE, os.path.join(EXPORT, "vocab.json"))
# speakers file if present
sp = os.path.join(CKPT_DIR, "speakers_xtts.pth")
# include a reference wav for inference conditioning
shutil.copy(SPEAKER_REFERENCE[0], os.path.join(EXPORT, "reference.wav"))

print("Export contents:", os.listdir(EXPORT))

api = HfApi()
api.create_repo(MODEL_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path=EXPORT, repo_id=MODEL_REPO, repo_type="model")
print("Pushed model to:", f"https://huggingface.co/{MODEL_REPO}")


## Notes
- **16 kHz source limit:** the lecture audio is 16 kHz, upsampled to 22.05 kHz for XTTS.
  The clone will be band-limited (no true high frequencies) but recognizable. Nothing
  to fix unless you can get a higher-rate source.
- **Tune `EPOCHS`:** watch the eval loss in the logs / TensorBoard. Stop when it plateaus
  to avoid overfitting to the lecturer's recording conditions.
- **Inference:** download `model.pth`, `config.json`, `vocab.json`, `reference.wav` from
  `MODEL_REPO` and run `inference/generate.py` on your CPU machine.
